# Three-Body Planetary Atom in Weber Electrodynamics

This notebook demonstrates **bound three-body states** in Weber electrodynamics:
two like charges forming a sub-critical nucleus, orbited by an unlike charge.

## Physics Summary

Weber's velocity-dependent force creates an effective inertial mass
$\mu_{\text{eff}}(r) = \mu(1 - \rho/r)$ that changes sign at the
**critical radius** $\rho = q_1 q_2 / (\mu c^2)$. For two like charges
($q_1 q_2 > 0$) with $r_0 < \rho$, the particles are permanently bound in
a "molecular" oscillation between $r_0$ and $r = 0$.

The **planetary atom** model adds an unlike charge orbiting this nucleus:

| Particle | Charge | Role |
|---|---|---|
| 1 | $+q$ | Nucleus (bound sub-critically with particle 2) |
| 2 | $+q$ | Nucleus (bound sub-critically with particle 1) |
| 3 | $-q$ | Orbiter (Coulomb-attracted to net charge $+2q$) |

The nucleus oscillates at period $T_{\text{nuc}} \approx 2\sqrt{2}\,r_{\text{nuc}}/c$
(fast), while the orbiter has period
$T_{\text{orb}} \approx 2\pi R / v_{\text{circ}}$ (slow). The natural
timescale separation $R \gg r_{\text{nuc}}$ keeps the orbiter far from the
nucleus, preventing three-body instability.

**Zöllner extension**: The mismatch parameter $a > 0$ strengthens unlike-pair
coupling by factor $(1+a)$, tightening and circularizing the orbiter's
trajectory. Like-pair coupling ($\kappa = 1$) is unaffected.

We perform four simulations:
1. **Run 1**: Pure Weber, near-circular orbit ($\eta = 0.7$, $a = 0$)
2. **Run 2**: Pure Weber, wider orbit ($\eta = 0.8$, $a = 0$)
3. **Run 3**: Zöllner-enhanced ($\eta = 0.8$, $a = 0.1$)
4. **Run 4**: Strong Zöllner circularization ($\eta = 0.8$, $a = 0.5$)

**References**: Weber, Sixth Memoir (1871) §§9.8–9.17; Frauenfelder & Weber,
*Anal. Math. Phys.* **14**:31 (2024); see
`docs/theory/CriticalRadiusAndLikeChargeAttraction.md`,
`docs/theory/InitialConditions.md`, and
`docs/exploratory/ThreeBodyBoundStates.md`.

In [ ]:
using WeberElectrodynamics
using LinearAlgebra
using Plots
using Printf

## 1. System Construction and Physical Parameters

In [ ]:
# Physical parameters
m = 1.0               # equal masses
q_pos = 1.0           # positive charges (nucleus)
q_neg = -1.0          # negative charge (orbiter)
c = 4.0

# Derived quantities
mu_nuc = m * m / (m + m)          # nucleus reduced mass = 0.5
rho = q_pos^2 / (mu_nuc * c^2)   # critical radius = 0.125
M_nuc = 2m                         # nucleus total mass
mu_orb = M_nuc * m / (M_nuc + m)  # orbiter reduced mass = 2/3

# Nucleus parameters
r_nuc = 0.05   # sub-critical nucleus separation
T_nuc = 2 * sqrt(2) * r_nuc / c   # nucleus oscillation period

# Orbiter parameters
R = 1.0        # orbiter distance from nucleus COM
Q_eff = 2.0    # effective charge seen by orbiter (q1 + q2)
v_circ = sqrt(Q_eff / (mu_orb * R))  # circular orbit speed
T_orb = 2 * pi * R / v_circ          # orbiter orbital period

# Integration parameters
dt = 1e-4
bounce_r = 0.02

system = WeberSystem(3, 2)

@printf("Three-body planetary atom:\n")
@printf("  Particles:  %d (2D)\n", system.n_particles)
@printf("  DOF:        %d\n", system.degrees_of_freedom)
@printf("\nNucleus (particles 1, 2):\n")
@printf("  q1 = q2 = +%.1f, m = %.1f\n", q_pos, m)
@printf("  r_nuc = %.4f < rho = %.4f\n", r_nuc, rho)
@printf("  T_nuc ~ %.6f\n", T_nuc)
@printf("\nOrbiter (particle 3):\n")
@printf("  q3 = %.1f, m3 = %.1f\n", q_neg, m)
@printf("  R = %.2f, mu_orb = %.4f\n", R, mu_orb)
@printf("  v_circ = %.4f, T_orb ~ %.4f\n", v_circ, T_orb)
@printf("  T_orb / T_nuc = %.1f  (timescale separation)\n", T_orb / T_nuc)
@printf("\nIntegration:\n")
@printf("  dt = %.0e, bounce_r = %.2f, c = %.1f\n", dt, bounce_r, c)

## 2. Initial Condition Construction

The planetary atom IC follows the recipe from `docs/theory/InitialConditions.md` §9:

1. **Nucleus** (particles 1, 2): placed at $\pm r_{\text{nuc}}/2$ along the
   $x$-axis with zero momenta (at the outer turning point of the sub-critical
   oscillation).
2. **Orbiter** (particle 3): placed at $(0, R)$ with tangential momentum
   $p_{3x} = m_3 \cdot \eta \cdot v_{\text{circ}}$.
3. **COM and momentum conservation**: all positions shifted to place COM at
   origin; nucleus momenta adjusted so $\sum_i \vec{p}_i = 0$.

In [ ]:
function make_planetary_atom_ic(r_nuc, R, eta_orb, m, q_pos, q_neg, c)
    # Positions (before COM adjustment)
    x1 = -r_nuc / 2;  y1 = 0.0
    x2 = +r_nuc / 2;  y2 = 0.0
    x3 = 0.0;          y3 = R

    # Orbiter circular speed estimate
    M_nuc = 2m
    mu_orb = M_nuc * m / (M_nuc + m)
    Q_eff = 2 * abs(q_pos * q_neg)  # |q1*q3| + |q2*q3|
    v_circ = sqrt(Q_eff / (mu_orb * R))
    v_orb = eta_orb * v_circ

    # Momenta: orbiter tangential (x-direction at (0, R))
    px3 = m * v_orb
    py3 = 0.0

    # Zero total momentum: distribute recoil to nucleus
    px1 = -px3 / 2;  py1 = 0.0
    px2 = -px3 / 2;  py2 = 0.0

    # COM correction
    M_total = 3m
    cx = (m * x1 + m * x2 + m * x3) / M_total
    cy = (m * y1 + m * y2 + m * y3) / M_total

    q0 = [x1 - cx, y1 - cy, x2 - cx, y2 - cy, x3 - cx, y3 - cy]
    p0 = [px1, py1, px2, py2, px3, py3]

    return q0, p0, v_circ, v_orb
end

# Helper: compute all pair separations from solution
function compute_pair_separations(sol)
    nt = length(sol.t)
    r12 = zeros(nt); r13 = zeros(nt); r23 = zeros(nt)
    for k in 1:nt
        qk = sol.q[k]
        r12[k] = sqrt((qk[1] - qk[3])^2 + (qk[2] - qk[4])^2)
        r13[k] = sqrt((qk[1] - qk[5])^2 + (qk[2] - qk[6])^2)
        r23[k] = sqrt((qk[3] - qk[5])^2 + (qk[4] - qk[6])^2)
    end
    return r12, r13, r23
end

# Helper: plot all pair separations
function plot_pair_separations(t, r12, r13, r23, rho, title_str)
    plt = plot(;
        title = title_str,
        xlabel = "Time t",
        ylabel = "Pair separation",
        legend = :outertopright,
        framestyle = :box,
        grid = true, gridalpha = 0.2,
        size = (1200, 500),
    )
    plot!(plt, t, r12, label = "r₁₂ (nucleus)", linewidth = 1.5, color = :steelblue)
    plot!(plt, t, r13, label = "r₁₃ (orb-nuc1)", linewidth = 1, color = :firebrick, alpha = 0.7)
    plot!(plt, t, r23, label = "r₂₃ (orb-nuc2)", linewidth = 1, color = :forestgreen, alpha = 0.7)
    hline!(plt, [rho], linestyle = :dash, linewidth = 2, color = :black,
        label = @sprintf("ρ = %.3f", rho))
    return plt
end

println("Helpers defined.")

## 3. Run 1: Pure Weber, Near-Circular Orbit ($\eta = 0.7$, $a = 0$)

At $\eta_{\text{orb}} = 0.7$, the orbiter speed is 70% of the circular speed.
This produces a nearly circular orbit with the orbiter distance varying
by only $\sim 5\%$ around $R = 1.0$.

In [ ]:
eta_1 = 0.7
tmax_1 = 100.0

q0_1, p0_1, vc_1, vorb_1 = make_planetary_atom_ic(r_nuc, R, eta_1, m, q_pos, q_neg, c)

prob_1 = WeberProblem(system, (0.0, tmax_1), q0_1, p0_1;
    masses = [m, m, m], charges = [q_pos, q_pos, q_neg], c = c, dt = dt,
    regularization_enabled = false,
    regularization_collision_bounce_radius = bounce_r,
    zollner_enabled = false, zollner_a = 0.0)

sol_1 = solve(prob_1)

@printf("Run 1: Pure Weber, eta = %.1f, a = 0\n", eta_1)
@printf("  retcode: %s, steps: %d\n", sol_1.retcode, length(sol_1.t))
@printf("  v_orb = %.4f (v_circ = %.4f)\n", vorb_1, vc_1)

In [ ]:
traj_1 = compute_trajectory_data(sol_1, 3, 2; stride = 1)
energy_1 = compute_energy_timeseries(sol_1; stride = 1)
momentum_1 = compute_momentum_timeseries(sol_1; stride = 1)
r12_1, r13_1, r23_1 = compute_pair_separations(sol_1)

@printf("Run 1 diagnostics:\n")
@printf("  E(0) = %.6f\n", energy_1.total_energy[1])
@printf("  Energy error (%%): %.4f\n", energy_1.statistics.global_error_percent_max)
@printf("  Nucleus r₁₂: [%.4e, %.4f]  (rho = %.4f) %s\n",
    minimum(r12_1), maximum(r12_1), rho,
    maximum(r12_1) < rho ? "INTACT" : "BROKEN")
@printf("  Orbiter r₁₃: [%.4f, %.4f]\n", minimum(r13_1), maximum(r13_1))
@printf("  Orbiter r₂₃: [%.4f, %.4f]\n", minimum(r23_1), maximum(r23_1))

### Run 1 Plots

In [ ]:
plot_trajectories(traj_1)

In [ ]:
plot_pair_separations(sol_1.t, r12_1, r13_1, r23_1, rho,
    "Run 1: Pair Separations (η=0.7, a=0)")

In [ ]:
plot_energy(energy_1)

In [ ]:
plot_energy_errors(energy_1)

In [ ]:
plot_momentum(momentum_1)

In [ ]:
# Phase space for the nucleus pair (1,2)
forces_12_1 = compute_pair_force_timeseries(sol_1, (1, 2), 3, 2,
    [m, m, m], [q_pos, q_pos, q_neg], c; stride = 1)
plot_phase_space(forces_12_1)

In [ ]:
# Phase space for an orbiter pair (1,3)
forces_13_1 = compute_pair_force_timeseries(sol_1, (1, 3), 3, 2,
    [m, m, m], [q_pos, q_pos, q_neg], c; stride = 1)
plot_phase_space(forces_13_1)

## 4. Run 2: Pure Weber, Wider Orbit ($\eta = 0.8$, $a = 0$)

At $\eta = 0.8$, the orbiter has more kinetic energy and traces a wider
elliptical orbit. The apoapsis extends to $\sim 2.5 R$.

In [ ]:
eta_2 = 0.8
tmax_2 = 100.0

q0_2, p0_2, vc_2, vorb_2 = make_planetary_atom_ic(r_nuc, R, eta_2, m, q_pos, q_neg, c)

prob_2 = WeberProblem(system, (0.0, tmax_2), q0_2, p0_2;
    masses = [m, m, m], charges = [q_pos, q_pos, q_neg], c = c, dt = dt,
    regularization_enabled = false,
    regularization_collision_bounce_radius = bounce_r,
    zollner_enabled = false, zollner_a = 0.0)

sol_2 = solve(prob_2)
energy_2 = compute_energy_timeseries(sol_2; stride = 1)
traj_2 = compute_trajectory_data(sol_2, 3, 2; stride = 1)
r12_2, r13_2, r23_2 = compute_pair_separations(sol_2)

@printf("Run 2: Pure Weber, eta = %.1f, a = 0\n", eta_2)
@printf("  retcode: %s, steps: %d\n", sol_2.retcode, length(sol_2.t))
@printf("  Energy error (%%): %.4f\n", energy_2.statistics.global_error_percent_max)
@printf("  Nucleus r₁₂: [%.4e, %.4f] %s\n",
    minimum(r12_2), maximum(r12_2),
    maximum(r12_2) < rho ? "INTACT" : "BROKEN")
@printf("  Orbiter r₁₃: [%.4f, %.4f]\n", minimum(r13_2), maximum(r13_2))

In [ ]:
plot_trajectories(traj_2)

In [ ]:
plot_pair_separations(sol_2.t, r12_2, r13_2, r23_2, rho,
    "Run 2: Pair Separations (η=0.8, a=0)")

In [ ]:
plot_energy(energy_2)

## 5. Run 3: Zöllner-Enhanced ($\eta = 0.8$, $a = 0.1$)

The Zöllner mismatch parameter $a = 0.1$ strengthens the unlike-pair coupling
by 10%. This tightens the orbiter's orbit, reducing the apoapsis from
$\sim 2.5 R$ to $\sim 1.8 R$.

In [ ]:
eta_3 = 0.8
a_3 = 0.1
tmax_3 = 100.0

q0_3, p0_3, vc_3, vorb_3 = make_planetary_atom_ic(r_nuc, R, eta_3, m, q_pos, q_neg, c)

prob_3 = WeberProblem(system, (0.0, tmax_3), q0_3, p0_3;
    masses = [m, m, m], charges = [q_pos, q_pos, q_neg], c = c, dt = dt,
    regularization_enabled = false,
    regularization_collision_bounce_radius = bounce_r,
    zollner_enabled = true, zollner_a = a_3)

sol_3 = solve(prob_3)
energy_3 = compute_energy_timeseries(sol_3; stride = 1)
traj_3 = compute_trajectory_data(sol_3, 3, 2; stride = 1)
r12_3, r13_3, r23_3 = compute_pair_separations(sol_3)

@printf("Run 3: Zöllner a = %.1f, eta = %.1f\n", a_3, eta_3)
@printf("  retcode: %s, steps: %d\n", sol_3.retcode, length(sol_3.t))
@printf("  Energy error (%%): %.4f\n", energy_3.statistics.global_error_percent_max)
@printf("  Nucleus r₁₂: [%.4e, %.4f] %s\n",
    minimum(r12_3), maximum(r12_3),
    maximum(r12_3) < rho ? "INTACT" : "BROKEN")
@printf("  Orbiter r₁₃: [%.4f, %.4f]\n", minimum(r13_3), maximum(r13_3))

In [ ]:
plot_trajectories(traj_3)

In [ ]:
plot_pair_separations(sol_3.t, r12_3, r13_3, r23_3, rho,
    "Run 3: Pair Separations (η=0.8, a=0.1)")

In [ ]:
plot_energy(energy_3)

## 6. Run 4: Strong Zöllner Circularization ($\eta = 0.8$, $a = 0.5$)

At $a = 0.5$, the Zöllner gravitational residual is strong enough to
transform the wide elliptical orbit ($a = 0$: range $[1.0, 2.5]$) into
a **nearly circular** orbit ($a = 0.5$: range $[0.85, 1.03]$). This
demonstrates the Zöllner circularization effect.

In [ ]:
eta_4 = 0.8
a_4 = 0.5
tmax_4 = 100.0

q0_4, p0_4, vc_4, vorb_4 = make_planetary_atom_ic(r_nuc, R, eta_4, m, q_pos, q_neg, c)

prob_4 = WeberProblem(system, (0.0, tmax_4), q0_4, p0_4;
    masses = [m, m, m], charges = [q_pos, q_pos, q_neg], c = c, dt = dt,
    regularization_enabled = false,
    regularization_collision_bounce_radius = bounce_r,
    zollner_enabled = true, zollner_a = a_4)

sol_4 = solve(prob_4)
energy_4 = compute_energy_timeseries(sol_4; stride = 1)
traj_4 = compute_trajectory_data(sol_4, 3, 2; stride = 1)
r12_4, r13_4, r23_4 = compute_pair_separations(sol_4)

@printf("Run 4: Zöllner a = %.1f, eta = %.1f\n", a_4, eta_4)
@printf("  retcode: %s, steps: %d\n", sol_4.retcode, length(sol_4.t))
@printf("  Energy error (%%): %.4f\n", energy_4.statistics.global_error_percent_max)
@printf("  Nucleus r₁₂: [%.4e, %.4f] %s\n",
    minimum(r12_4), maximum(r12_4),
    maximum(r12_4) < rho ? "INTACT" : "BROKEN")
@printf("  Orbiter r₁₃: [%.4f, %.4f]\n", minimum(r13_4), maximum(r13_4))

In [ ]:
plot_trajectories(traj_4)

In [ ]:
plot_pair_separations(sol_4.t, r12_4, r13_4, r23_4, rho,
    "Run 4: Pair Separations (η=0.8, a=0.5)")

In [ ]:
plot_energy(energy_4)

## 7. Comparative Analysis

### Trajectory Overlay: Zöllner Circularization Effect

In [ ]:
# Overlay orbiter trajectories for Runs 2-4 (all eta=0.8, varying a)
plt_overlay = plot(;
    title = "Orbiter Trajectory: Zöllner Circularization (η = 0.8)",
    xlabel = "x", ylabel = "y",
    aspect_ratio = :equal,
    legend = :outertopright,
    framestyle = :box,
    grid = true, gridalpha = 0.2,
    size = (1000, 1000),
)

# Orbiter is particle 3 (indices 5, 6 in q vector)
x3_2 = [sol_2.q[k][5] for k in 1:10:length(sol_2.t)]
y3_2 = [sol_2.q[k][6] for k in 1:10:length(sol_2.t)]
x3_3 = [sol_3.q[k][5] for k in 1:10:length(sol_3.t)]
y3_3 = [sol_3.q[k][6] for k in 1:10:length(sol_3.t)]
x3_4 = [sol_4.q[k][5] for k in 1:10:length(sol_4.t)]
y3_4 = [sol_4.q[k][6] for k in 1:10:length(sol_4.t)]

plot!(plt_overlay, x3_2, y3_2, label = "a = 0 (Weber)",
    linewidth = 1, color = :steelblue, alpha = 0.6)
plot!(plt_overlay, x3_3, y3_3, label = "a = 0.1 (Zöllner)",
    linewidth = 1, color = :forestgreen, alpha = 0.6)
plot!(plt_overlay, x3_4, y3_4, label = "a = 0.5 (strong Zöllner)",
    linewidth = 1.5, color = :firebrick)

# Mark nucleus location
scatter!(plt_overlay, [0.0], [0.0], marker = :star5, markersize = 8,
    color = :gold, label = "Nucleus COM")

plt_overlay

### Orbiter Separation Comparison

In [ ]:
# Orbiter distance from nucleus COM
function orbiter_distance_from_com(sol)
    nt = length(sol.t)
    d = zeros(nt)
    for k in 1:nt
        qk = sol.q[k]
        cx = (qk[1] + qk[3]) / 2  # nucleus COM x
        cy = (qk[2] + qk[4]) / 2  # nucleus COM y
        d[k] = sqrt((qk[5] - cx)^2 + (qk[6] - cy)^2)
    end
    return d
end

d_orb_1 = orbiter_distance_from_com(sol_1)
d_orb_2 = orbiter_distance_from_com(sol_2)
d_orb_3 = orbiter_distance_from_com(sol_3)
d_orb_4 = orbiter_distance_from_com(sol_4)

plt_dorb = plot(;
    title = "Orbiter Distance from Nucleus COM",
    xlabel = "Time t", ylabel = "Distance",
    legend = :outertopright,
    framestyle = :box,
    grid = true, gridalpha = 0.2,
    size = (1200, 500),
)
stride = max(1, length(sol_1.t) ÷ 2000)
plot!(plt_dorb, sol_1.t[1:stride:end], d_orb_1[1:stride:end],
    label = "Run 1: η=0.7, a=0", linewidth = 1.5, color = :purple)
plot!(plt_dorb, sol_2.t[1:stride:end], d_orb_2[1:stride:end],
    label = "Run 2: η=0.8, a=0", linewidth = 1.5, color = :steelblue)
stride3 = max(1, length(sol_3.t) ÷ 2000)
plot!(plt_dorb, sol_3.t[1:stride3:end], d_orb_3[1:stride3:end],
    label = "Run 3: η=0.8, a=0.1", linewidth = 1.5, color = :forestgreen)
stride4 = max(1, length(sol_4.t) ÷ 2000)
plot!(plt_dorb, sol_4.t[1:stride4:end], d_orb_4[1:stride4:end],
    label = "Run 4: η=0.8, a=0.5", linewidth = 1.5, color = :firebrick)
hline!(plt_dorb, [R], linestyle = :dash, linewidth = 1, color = :black,
    label = @sprintf("R = %.1f (initial)", R))

plt_dorb

### Nucleus Integrity Check

In [ ]:
plt_nuc = plot(;
    title = "Nucleus Separation r₁₂(t): All Runs",
    xlabel = "Time t", ylabel = "r₁₂",
    legend = :outertopright,
    framestyle = :box,
    grid = true, gridalpha = 0.2,
    size = (1200, 400),
)

plot!(plt_nuc, sol_1.t[1:stride:end], r12_1[1:stride:end],
    label = "Run 1", linewidth = 1, color = :purple, alpha = 0.7)
plot!(plt_nuc, sol_2.t[1:stride:end], r12_2[1:stride:end],
    label = "Run 2", linewidth = 1, color = :steelblue, alpha = 0.7)
plot!(plt_nuc, sol_3.t[1:stride3:end], r12_3[1:stride3:end],
    label = "Run 3", linewidth = 1, color = :forestgreen, alpha = 0.7)
plot!(plt_nuc, sol_4.t[1:stride4:end], r12_4[1:stride4:end],
    label = "Run 4", linewidth = 1, color = :firebrick, alpha = 0.7)
hline!(plt_nuc, [rho], linestyle = :dash, linewidth = 2, color = :black,
    label = @sprintf("ρ = %.3f", rho))
hline!(plt_nuc, [r_nuc], linestyle = :dot, linewidth = 1, color = :gray,
    label = @sprintf("r_nuc = %.2f", r_nuc))

plt_nuc

## 8. Summary and Diagnostics

In [ ]:
println("=" ^ 90)
println("THREE-BODY PLANETARY ATOM -- SUMMARY")
println("=" ^ 90)

@printf("\nPhysical parameters:\n")
@printf("  m = %.1f (all), q = +%.1f/+%.1f/-%.1f, c = %.1f\n", m, q_pos, q_pos, abs(q_neg), c)
@printf("  Critical radius rho = %.4f\n", rho)
@printf("  Nucleus: r_nuc = %.4f, T_nuc ~ %.6f\n", r_nuc, T_nuc)
@printf("  Orbiter: R = %.2f, v_circ = %.4f, T_orb ~ %.4f\n", R, v_circ, T_orb)
@printf("  dt = %.0e, bounce_r = %.2f\n", dt, bounce_r)

println("\n" * "-" ^ 90)
@printf("%-8s  %-5s  %-5s  %-8s  %-9s  %-15s  %-15s  %s\n",
    "", "η", "a", "retcode", "E_err(%)", "nuc r₁₂ max", "orb range", "Status")
println("-" ^ 90)

for (label, eta, a, sol, en, r12, r13, r23) in [
    ("Run 1", eta_1, 0.0, sol_1, energy_1, r12_1, r13_1, r23_1),
    ("Run 2", eta_2, 0.0, sol_2, energy_2, r12_2, r13_2, r23_2),
    ("Run 3", eta_3, a_3, sol_3, energy_3, r12_3, r13_3, r23_3),
    ("Run 4", eta_4, a_4, sol_4, energy_4, r12_4, r13_4, r23_4),
]
    max_orb = max(maximum(r13), maximum(r23))
    min_orb = min(minimum(r13), minimum(r23))
    nuc_ok = maximum(r12) < rho
    @printf("%-8s  %.1f    %.1f    %-8s  %8.4f   %.4f          [%.2f, %.2f]       %s\n",
        label, eta, a, sol.retcode, en.statistics.global_error_percent_max,
        maximum(r12), min_orb, max_orb,
        nuc_ok ? "BOUND" : "BROKEN")
end

println("-" ^ 90)
@printf("\nZöllner circularization (η = 0.8):\n")
@printf("  a = 0.0: orbiter range [%.2f, %.2f]\n", min(minimum(r13_2), minimum(r23_2)), max(maximum(r13_2), maximum(r23_2)))
@printf("  a = 0.1: orbiter range [%.2f, %.2f]\n", min(minimum(r13_3), minimum(r23_3)), max(maximum(r13_3), maximum(r23_3)))
@printf("  a = 0.5: orbiter range [%.2f, %.2f]  <-- near-circular\n", min(minimum(r13_4), minimum(r23_4)), max(maximum(r13_4), maximum(r23_4)))

## 9. Conclusions

This notebook demonstrated the **three-body planetary atom** in Weber
electrodynamics — a bound state consisting of a sub-critical like-charge
nucleus orbited by an unlike charge.

### Key Findings

1. **The planetary atom is stable**: The nucleus (two positive charges
   oscillating sub-critically) remains intact while the negative orbiter
   traces bound orbits for 100+ time units (~28 orbiter periods).

2. **Orbiter binding range**: $\eta_{\text{orb}} \in [0.5, 0.9]$ produces
   bound orbits. Below $\eta = 0.5$, the orbiter disrupts the nucleus;
   above $\eta = 1.0$, the orbiter escapes.

3. **Zöllner circularization**: The mismatch parameter $a$ progressively
   tightens the orbiter's orbit. At $\eta = 0.8$:
   - $a = 0$: wide ellipse, range $[1.0, 2.5]$
   - $a = 0.1$: moderate, range $[1.0, 1.8]$
   - $a = 0.5$: near-circular, range $[0.85, 1.03]$

4. **Timescale separation is essential**: The nucleus oscillation period
   ($T_{\text{nuc}} \approx 0.035$) is 100× faster than the orbiter period
   ($T_{\text{orb}} \approx 3.6$). This keeps the orbiter far from the nucleus
   during most of its orbit, preventing three-body instability.

5. **Three like charges (+++) do not form stable bound states**: The symmetric
   three-body collapse overwhelms the integrator, and no configuration
   survives beyond a few oscillation periods.

### Physical Significance

This realizes Weber's "planetary model of the atom" (Sixth Memoir, §9.17):
a dyad of like charges bound by the velocity-dependent force below the
critical radius, orbited by an unlike charge. The Zöllner extension provides
an additional gravitational-like binding that circularizes the orbit —
analogous to tidal circularization in celestial mechanics.

See `docs/exploratory/ThreeBodyBoundStates.md` for the full parameter study.